# ORC Basics — 09: ORC to Parquet Migration

Migrating from ORC to Parquet for better ecosystem compatibility.

Topics: reading ORC, writing Parquet, schema preservation, validation,
partition migration, incremental migration pattern.


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/orc_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('orc-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 13:30:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


## Step 1 — Setup: ORC Data Lake



In [2]:

import random,datetime,pathlib
random.seed(42); N=300_000
ORC_LAKE=f"{DATA_DIR}/orc_lake"
CATS=["Electronics","Clothing","Books","Food","Sports"]
CTRS=["US","UK","DE","FR","JP"]

for cat in CATS:
    rows=[]
    base=datetime.date(2023,1,1)
    for i in range(N//len(CATS)):
        d=base+datetime.timedelta(days=random.randint(0,365))
        rows.append((i+1,random.randint(1,5000),cat,random.choice(CTRS),
                     random.randint(1,10),round(random.uniform(5,1000),2),d))
    df_cat=spark.createDataFrame(rows,["order_id","customer_id","category","country","quantity","price","order_date"])
    df_cat.write.mode("overwrite").option("compression","zlib") \
          .orc(f"{ORC_LAKE}/category={cat}")
    print(f"  Written {len(rows):,} rows → category={cat}/")

total=spark.read.orc(ORC_LAKE).count()
print(f"\nORC lake total: {total:,} rows across {len(CATS)} partitions")


  Written 60,000 rows → category=Electronics/
  Written 60,000 rows → category=Clothing/
  Written 60,000 rows → category=Books/
  Written 60,000 rows → category=Food/
  Written 60,000 rows → category=Sports/


26/04/10 13:31:15 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `category` already exists. Choose another name or rename the existing column. SQLSTATE: 42711
[Stage 5:>                                                          (0 + 4) / 4]


ORC lake total: 300,000 rows across 5 partitions


## Step 2 — Read ORC, Write Parquet (Schema Preserved)



In [3]:

PQ_LAKE=f"{DATA_DIR}/parquet_lake"
t0=time.time()
df_orc = spark.read.orc(ORC_LAKE)
df_orc.write.mode("overwrite") \
            .partitionBy("category") \
            .option("compression","zstd") \
            .parquet(PQ_LAKE)
t_migrate = time.time()-t0

orc_mb=sum(f.stat().st_size for f in pathlib.Path(ORC_LAKE).rglob("*.orc"))/1024/1024
pq_mb =sum(f.stat().st_size for f in pathlib.Path(PQ_LAKE).rglob("*.parquet"))/1024/1024
print(f"Migration: {t_migrate:.1f}s")
print(f"ORC size : {orc_mb:.1f} MB  (zlib compressed)")
print(f"Parquet  : {pq_mb:.1f} MB  (zstd compressed)")
print(f"Size ratio: {pq_mb/orc_mb:.2f}x  (Parquet+zstd vs ORC+zlib)")


26/04/10 13:31:18 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `category` already exists. Choose another name or rename the existing column. SQLSTATE: 42711
                                                                                

Migration: 4.2s
ORC size : 2.2 MB  (zlib compressed)
Parquet  : 2.6 MB  (zstd compressed)
Size ratio: 1.16x  (Parquet+zstd vs ORC+zlib)


## Step 3 — Validation



In [4]:

orc_count = spark.read.orc(ORC_LAKE).count()
pq_count  = spark.read.parquet(PQ_LAKE).count()
print(f"Row count validation:")
print(f"  ORC     : {orc_count:,}")
print(f"  Parquet : {pq_count:,}")
print(f"  Match   : {'✅ PASS' if orc_count==pq_count else '❌ FAIL'}")
print()

# Numeric checksum
for col_name in ["quantity","price"]:
    orc_sum = spark.read.orc(ORC_LAKE).agg(F.sum(col_name)).collect()[0][0]
    pq_sum  = spark.read.parquet(PQ_LAKE).agg(F.sum(col_name)).collect()[0][0]
    match = abs((orc_sum or 0)-(pq_sum or 0)) < 0.01
    print(f"  {col_name} sum: ORC={orc_sum:,.2f}  PQ={pq_sum:,.2f}  {'✅' if match else '❌'}")


26/04/10 13:31:22 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `category` already exists. Choose another name or rename the existing column. SQLSTATE: 42711


Row count validation:
  ORC     : 300,000
  Parquet : 300,000
  Match   : ✅ PASS



26/04/10 13:31:24 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `category` already exists. Choose another name or rename the existing column. SQLSTATE: 42711


  quantity sum: ORC=1,647,187.00  PQ=1,647,187.00  ✅


26/04/10 13:31:26 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `category` already exists. Choose another name or rename the existing column. SQLSTATE: 42711


  price sum: ORC=150,843,244.24  PQ=150,843,244.24  ✅


## Step 4 — Query Performance After Migration



In [5]:

queries=[
    ("Full scan sum",   lambda p,f: spark.read.format(f).load(p).agg(F.sum("price")).collect()),
    ("Partition filter",lambda p,f: spark.read.format(f).load(p).filter(F.col("category")=="Electronics").count()),
    ("GroupBy",         lambda p,f: spark.read.format(f).load(p).groupBy("country").agg(F.sum("price")).collect()),
]
print(f"{'Query':<25} {'ORC':>8} {'Parquet':>10} {'Improvement':>13}")
print("-"*60)
for label,fn in queries:
    to,tp=[],[]
    for _ in range(3):
        t0=time.time(); fn(ORC_LAKE,"orc");    to.append(time.time()-t0)
        t0=time.time(); fn(PQ_LAKE,"parquet"); tp.append(time.time()-t0)
    t_o,t_p=sorted(to)[1],sorted(tp)[1]
    diff=f"{t_o/t_p:.1f}x {'faster' if t_p<t_o else 'slower'}"
    print(f"  {label:<23} {t_o:>6.3f}s  {t_p:>8.3f}s  {diff:>13}")


Query                          ORC    Parquet   Improvement
------------------------------------------------------------


26/04/10 13:31:28 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `category` already exists. Choose another name or rename the existing column. SQLSTATE: 42711
26/04/10 13:31:30 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `category` already exists. Choose another name or rename the existing column. SQLSTATE: 42711
26/04/10 13:31:31 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `category` already exists. Choose another name or rename the existing column. SQLSTATE: 42711


  Full scan sum            0.906s     0.855s    1.1x faster


26/04/10 13:31:33 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `category` already exists. Choose another name or rename the existing column. SQLSTATE: 42711
26/04/10 13:31:34 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `category` already exists. Choose another name or rename the existing column. SQLSTATE: 42711
26/04/10 13:31:35 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `category` already exists. Choose another name or rename the existing column. SQLSTATE: 42711


  Partition filter         0.625s     0.555s    1.1x faster


26/04/10 13:31:37 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `category` already exists. Choose another name or rename the existing column. SQLSTATE: 42711
26/04/10 13:31:39 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `category` already exists. Choose another name or rename the existing column. SQLSTATE: 42711
26/04/10 13:31:40 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `category` already exists. Choose another name or rename the existing column. SQLSTATE: 42711


  GroupBy                  0.974s     0.853s    1.1x faster


## Summary



In [6]:

print("""
ORC → Parquet migration pattern:
  1. Read ORC (schema auto-detected)
  2. Optionally add derived columns / normalize types
  3. Write Parquet (partitioned by same columns)
  4. Validate: row count + numeric checksums
  5. Update downstream pipelines to use Parquet path
  6. Archive or delete ORC after validation

Why migrate to Parquet?
  ✅ Wider ecosystem (Delta, Iceberg, Trino, BigQuery, Athena)
  ✅ ZSTD compression typically smaller than ORC+zlib
  ✅ Better tooling (pyarrow, pandas, DuckDB)
  ✅ Default for modern table formats
""")



ORC → Parquet migration pattern:
  1. Read ORC (schema auto-detected)
  2. Optionally add derived columns / normalize types
  3. Write Parquet (partitioned by same columns)
  4. Validate: row count + numeric checksums
  5. Update downstream pipelines to use Parquet path
  6. Archive or delete ORC after validation

Why migrate to Parquet?
  ✅ Wider ecosystem (Delta, Iceberg, Trino, BigQuery, Athena)
  ✅ ZSTD compression typically smaller than ORC+zlib
  ✅ Better tooling (pyarrow, pandas, DuckDB)
  ✅ Default for modern table formats

